In [ ]:
####KEGG GO enrich
library(Seurat)
library(patchwork)
library(dplyr)
library(glue)
library(future)
library(ggplot2)
library(cowplot)
options(future.globals.maxSize = 30 * 1024^3)
library(stringr)
library(pheatmap)
library(RColorBrewer)
library(ggpubr)
library(viridis)
library(ggrepel)
library(reshape2)
library(tidyr)
library(clusterProfiler)

In [1]:
gene <- read.csv("regions_gene.csv")

In [3]:
head(gene)

,gene,Regions
,<chr>,<chr>
1,2200002D01Rik,Cornea&Conj
2,Aldh3a1,Cornea&Conj
3,Anxa2,Cornea&Conj
4,Aqp3,Cornea&Conj
5,Aqp5,Cornea&Conj
6,Asprv1,Cornea&Conj


In [9]:
library(clusterProfiler)

In [ ]:
n=0
for(i in unique(gene$Regions)){
    n=n+1
    gene1 <- subset(gene,subset = Regions==i)
    m <- as.data.frame(table(gene1$gene))
    colnames(m) <- c("gene","num")
    m1 <- subset(m,subset = num>=1)
    m1$divi_area <- i
    df_id <- bitr(as.vector(m1$gene),
                  fromType  = "SYMBOL",
                  toType = "ENTREZID",
                  OrgDb = "org.Mm.eg.db")
    GO <- enrichGO(OrgDb = "org.Mm.eg.db", gene = df_id$ENTREZID, pvalueCutoff = 0.05, readable = TRUE,ont = "BP",keyType = "ENTREZID")
    GO_output <- GO@result[GO@result$p.adjust < 0.05,]
    GO_result <- subset(GO_output,subset = Count>=3)
    dt <- GO_result
    dt1 <- dt%>% top_n(n = 10, wt = -p.adjust)
    dt1 <- as.data.frame(dt1)
    dt1$geneRatio <- sapply(dt1$GeneRatio, function(x) {
        parts <- unlist(strsplit(x, "/")) 
        as.numeric(parts[1]) / as.numeric(parts[2])
    })
    dt1$Regions <- i
    if(n==1){
        mer <- dt1
        m2 <- m1
    }
    else{
        mer <- rbind(mer,dt1)
        m2 <- rbind(m2,m1)
    }
}

In [18]:
head(mer)

,ID,Description,GeneRatio,BgRatio,RichFactor,FoldEnrichment,zScore,pvalue,p.adjust,qvalue,geneID,Count,geneRatio,Regions
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>,<dbl>,<chr>
GO:0045104,GO:0045104,intermediate filament cytoskeleton organization,6/35,96/28928,0.06250000,51.65714,17.30328,1.711927e-09,6.016662e-07,4.030300e-07,Dsp/Krt12/Krt5/Krt6a/Krt6b/Ppl,6,0.1714286,Cornea&Conj
GO:0045103,GO:0045103,intermediate filament-based process,6/35,97/28928,0.06185567,51.12459,17.21061,1.823231e-09,6.016662e-07,4.030300e-07,Dsp/Krt12/Krt5/Krt6a/Krt6b/Ppl,6,0.1714286,Cornea&Conj
GO:0031424,GO:0031424,keratinization,5/35,64/28928,0.07812500,64.57143,17.71997,1.393741e-08,3.066231e-06,2.053934e-06,Krt5/Krt6a/Krt6b/Ppl/Sfn,5,0.1428571,Cornea&Conj
GO:0045109,GO:0045109,intermediate filament organization,5/35,76/28928,0.06578947,54.37594,16.21637,3.342280e-08,5.514762e-06,3.694099e-06,Dsp/Krt12/Krt5/Krt6a/Krt6b,5,0.1428571,Cornea&Conj
GO:0030216,GO:0030216,keratinocyte differentiation,6/35,169/28928,0.03550296,29.34370,12.86182,5.129768e-08,6.771294e-06,4.535795e-06,Dsp/Krt5/Krt6a/Krt6b/Ppl/Sfn,6,0.1714286,Cornea&Conj
GO:0043588,GO:0043588,skin development,7/35,351/28928,0.01994302,16.48319,10.15769,1.831342e-07,2.014476e-05,1.349410e-05,Asprv1/Dsp/Krt5/Krt6a/Krt6b/Ppl/Sfn,7,0.2000000,Cornea&Conj


In [19]:
write.csv(mer,"regions_GOBP.csv")